# KV-cache compression with ILF-diversified selection and mixed precision

This notebook implements and evaluates the project proposal end to end:

| Proposal item | Where |
|---|---|
| Attention scores $a_{i,l}$, layer frequency $d_i$, $\mathrm{ILF}_i$, diversified score $S_{i,l}=a_{i,l}(1+\lambda\,\mathrm{ILF}_i)$ | §3 and §4 |
| Dynamic selection (evict tokens) with plain attention ($\lambda=0$) vs ILF ($\lambda>0$) | §4 |
| ILF-guided mixed precision, {16, 8, 4, 0} bits per token, with **real packed storage** | §4 and §5 |
| Matched memory budgets plus uniform-quantization baselines and ablations | §8 |
| MMLU (general reasoning) | §10 |
| LongBench (long context) | §11 |
| Long-range retrieval with a **reused** prefix cache (many queries per compressed prefix) | §12 |
| Bytes-retained %, measured GPU memory, decode latency at batch 4 × 4096 tokens | §13 |
| Unique retained positions across layers (diversity), λ sweep | §14 |
| Several models (Qwen 2.5, Llama 2, Llama 3) | §16 |

**How to run.** Use a Colab GPU runtime and run top to bottom. `SMOKE_TEST = True` (§2) runs
every experiment on a handful of examples in a few minutes to check the pipeline. Set it to
`False` for real numbers. Results are saved as CSV/PNG under `results/<model>/`.

**Implementation notes.**
* A custom attention backend (`kv_compress`, SDPA-based) records $a_{i,l}$ during prefill:
  the attention mass that the last `OBS_WINDOW` prefix queries put on each token, averaged
  over heads (SnapKV-style). That avoids `output_attentions`, whose quadratic memory does not
  fit at 4k tokens. The backend also builds its own causal mask, so it stays correct after
  eviction, when cache length ≠ position index.
* Keys are cached after RoPE, and new tokens are fed with their *original* position ids, so
  eviction never shifts positions.
* The first `SINK_TOKENS` and last `OBS_WINDOW` prefix tokens always stay at 16 bits; only
  the tokens between them are scored, evicted or quantized. Their bytes count against the budget.
* Quantization is asymmetric min-max with fp16 scale and zero: per channel over 32-token groups for keys
  (keys have outlier channels, as in KIVI), per token over 32-channel groups for values. 4-bit
  codes are packed two per byte. Quantized tokens are dequantized on the fly at every decode step.
* **Prefix-caching setup.** With `QUERY_AWARE = False`, LongBench prompts are split where the
  question starts. The document prefix is compressed without seeing the question, and the
  question is then run on top of the compressed cache. This matches the proposal's
  motivation (one cached prefix serving many future queries). Set `True` for the usual
  query-aware SnapKV-style protocol.


## 1. Dependencies
If installing changes a package already imported in your kernel, restart the kernel before continuing.


In [ ]:
%pip install -q "torch>=2.6" "transformers==4.57.1" "accelerate>=0.26.0" "lm_eval[hf,longbench]==0.4.11" sentencepiece pandas matplotlib

import json, math, time, random, re, gc
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
import pandas as pd
from importlib.metadata import version

print({name: version(name) for name in ["torch", "transformers", "lm_eval"]})


## 2. Experiment configuration
`BUDGETS` are fractions of the fp16 KV-cache bytes. `"int4"` / `"int8"` resolve, for each input, to
exactly the bytes that uniform 4/8-bit quantization uses, which gives byte-matched comparisons.


In [ ]:
SMOKE_TEST = True   # tiny settings that run end to end in minutes; set False for real experiments

MODEL_ID = "Qwen/Qwen2.5-0.5B"
# Generalisation sweep (final section). Llama checkpoints are gated: request access on
# Hugging Face and run `from huggingface_hub import login; login()` first.
EXTRA_MODEL_IDS = ["Qwen/Qwen2.5-1.5B", "meta-llama/Meta-Llama-3-8B", "meta-llama/Llama-2-7b-hf"]

# Compression hyperparameters
SINK_TOKENS = 4
OBS_WINDOW = 32        # last W prefix tokens: always kept at 16 bits, and their queries give a_{i,l}
POOL_KERNEL = 7
LAMBDA = 1.0           # main ILF weight; LAMBDA_SWEEP explores it
LAMBDA_SWEEP = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0]
TIER_SPLIT = (1 / 3, 1 / 3, 1 / 3)  # budget share for the 16 / 8 / 4-bit tiers in mixed precision
# Memory budgets as a fraction of fp16 KV bytes. "int4"/"int8" match uniform quantization exactly.
BUDGETS = [0.1, 0.2, "int4", "int8"]
QUERY_AWARE = False    # False: compress the prefix before the question is seen (prefix caching)

MAX_INPUT_TOKENS = 4096
MMLU_TASKS, MMLU_LIMIT, MMLU_FEWSHOT = ["mmlu"], 10, 5   # limit applies per subject (57 subjects)
LONGBENCH_TASKS = ["longbench_hotpotqa", "longbench_2wikimqa", "longbench_multifieldqa_en",
                   "longbench_qasper", "longbench_passage_retrieval_en"]
LONGBENCH_LIMIT = 50
RETRIEVAL_DOCS, RETRIEVAL_FACTS, RETRIEVAL_TOKENS = 10, 20, 3000
BENCH_BATCH, BENCH_SEQ_LEN, BENCH_DECODE_TOKENS = 4, 4096, 64

if SMOKE_TEST:
    MMLU_TASKS, MMLU_LIMIT, MMLU_FEWSHOT = ["mmlu_abstract_algebra"], 4, 0
    LONGBENCH_TASKS, LONGBENCH_LIMIT = ["longbench_hotpotqa"], 2
    MAX_INPUT_TOKENS = 2048
    RETRIEVAL_DOCS, RETRIEVAL_FACTS, RETRIEVAL_TOKENS = 1, 6, 800
    BENCH_BATCH, BENCH_SEQ_LEN, BENCH_DECODE_TOKENS = 1, 512, 8
    BUDGETS = [0.2, "int4"]
    LAMBDA_SWEEP = [0.0, 1.0]

RESULTS_DIR = Path("results")
set_seed(42)


## 3. Attention backend: recording $a_{i,l}$
`kv_compression_attention` replaces eager attention. While `RECORDER` is active (only during the prefill
of a compressed method), each layer stores
$a_{i,l} = \operatorname{mean}_{h,\,q \in \text{window}} \operatorname{softmax}(q K^\top/\sqrt{d})_i$.


In [ ]:
import math
import torch.nn.functional as F
from transformers import AttentionInterface


class AttentionScoreRecorder:
    """While active, stores a_{i,l} for every layer l: attention from the last `window`
    query positions to every key i, averaged over query heads and window queries."""

    def __init__(self):
        self.active, self.window, self.scores = False, 32, {}

    def start(self, window):
        self.active, self.window, self.scores = True, window, {}

    def stop(self):
        scores, self.active, self.scores = self.scores, False, {}
        return scores


RECORDER = AttentionScoreRecorder()


def _window_attention(query, key, scaling, window):
    """[b, Hq, q, d] x [b, Hkv, k, d] -> [b, k] attention mass from the last `window` queries.
    Grouped-query heads are handled without materialising repeated keys."""
    b, hq, q_len, d = query.shape
    hk, k_len = key.shape[1], key.shape[2]
    n_rep, w = hq // hk, min(window, q_len)
    q = query[:, :, -w:].float().reshape(b, hk, n_rep * w, d)  # rows ordered (rep, window pos)
    logits = torch.matmul(q, key.float().transpose(-1, -2)) * scaling
    q_pos = (torch.arange(w, device=query.device) + (k_len - w)).repeat(n_rep)
    future = torch.arange(k_len, device=query.device)[None, :] > q_pos[:, None]
    logits.masked_fill_(future, float("-inf"))
    return logits.softmax(-1).mean(dim=(1, 2))


def kv_compression_attention(module, query, key, value, attention_mask, scaling=None, dropout=0.0, **kwargs):
    """SDPA attention that (1) optionally records window attention scores and
    (2) builds its own mask, so it works on caches whose length no longer equals the
    position index (after eviction). Assumes unpadded batches (no left padding)."""
    scaling = scaling if scaling is not None else 1.0 / math.sqrt(query.shape[-1])
    if RECORDER.active:
        RECORDER.scores[module.layer_idx] = _window_attention(query, key, scaling, RECORDER.window)
    n_rep = query.shape[1] // key.shape[1]
    if n_rep > 1:
        key = key.repeat_interleave(n_rep, dim=1)
        value = value.repeat_interleave(n_rep, dim=1)
    q_len, k_len = query.shape[2], key.shape[2]
    if q_len == 1:  # decoding: every retained cache entry is visible
        out = F.scaled_dot_product_attention(query, key, value, scale=scaling)
    elif q_len == k_len:  # prefill with an empty cache
        out = F.scaled_dot_product_attention(query, key, value, is_causal=True, scale=scaling)
    else:  # several new tokens on top of a (compressed) cache: causal among the new tokens
        q_pos = torch.arange(k_len - q_len, k_len, device=query.device)
        allowed = torch.arange(k_len, device=query.device)[None, :] <= q_pos[:, None]
        out = F.scaled_dot_product_attention(query, key, value, attn_mask=allowed, scale=scaling)
    return out.transpose(1, 2).contiguous(), None


AttentionInterface.register("kv_compress", kv_compression_attention)
print("Registered attention backend 'kv_compress'.")


## 4. ILF scoring and bit allocation
For every layer $l$ and candidate token $i$:

$$d_i=\sum_{l=1}^{L}\mathbf 1[a_{i,l}>\tau_l],\qquad \mathrm{ILF}_i=\log\frac{L+1}{d_i+1},\qquad S_{i,l}=a_{i,l}\,(1+\lambda\,\mathrm{ILF}_i)$$

By default $\tau_l$ is layer $l$'s $(1-k)$ quantile of attention, where $k$ is the fraction of tokens the
budget lets us keep. So $d_i$ counts the layers in which plain attention selection would have kept
token $i$. Tokens that every layer already wants get a small ILF boost; tokens that only a few layers
find important get a larger one.

* **select**: per layer, the top tokens by $S_{i,l}$ are kept at 16 bits; the rest are evicted.
* **mixed**: per layer, tokens ranked by $S_{i,l}$ fill a 16-bit tier, then an 8-bit tier, then a 4-bit
  tier; the rest get 0 bits (evicted). `TIER_SPLIT` sets how the byte budget is shared across tiers.
* $\lambda = 0$ recovers the plain attention-based baselines.


In [ ]:
from dataclasses import dataclass
from typing import Optional, Union


@dataclass(frozen=True)
class KVCompressionConfig:
    """One compression method.

    method:  "full"   - no compression (baseline)
             "select" - dynamic selection: keep the top tokens at 16 bits, evict the rest
             "quant"  - uniform quantization of every token to `quant_bits`
             "mixed"  - mixed precision: tokens ranked by score get 16 / 8 / 4 / 0 bits
    budget:  fraction of the fp16 KV-cache bytes to keep, or "int4"/"int8" to match the
             byte footprint of uniform 4/8-bit quantization on the same input.
    lam:     lambda in S_{i,l} = a_{i,l} (1 + lambda * ILF_i). lam = 0 is plain attention scoring.
    """

    name: str
    method: str = "full"
    budget: Union[float, str] = 1.0
    lam: float = 0.0
    quant_bits: int = 8
    tier_split: tuple = (1 / 3, 1 / 3, 1 / 3)  # share of the budget spent on 16 / 8 / 4-bit tiers
    sink_tokens: int = 4    # first tokens, always kept at 16 bits (attention sinks)
    window: int = 32        # last tokens, always kept at 16 bits; their queries score the rest
    pool_kernel: int = 7    # 1-D average pooling of scores along tokens (SnapKV-style); 1 disables
    tau_quantile: Optional[float] = None  # per-layer tau_l quantile; None -> 1 - keep fraction


def bytes_ratio(bits, head_dim, group=32):
    """Approximate bytes per stored token relative to fp16, including the fp16 scale + zero
    stored for every group of `group` elements (keys and values alike)."""
    if bits == 16:
        return 1.0
    if bits == 0:
        return 0.0
    return (bits / 8 + 4 / min(group, head_dim)) / 2


def resolve_budget(cfg, n_tokens, n_forced, head_dim):
    if isinstance(cfg.budget, str):  # "int4" / "int8": match uniform quantization
        bits = int(cfg.budget.removeprefix("int"))
        return (n_forced + (n_tokens - n_forced) * bytes_ratio(bits, head_dim)) / n_tokens
    return float(cfg.budget)


def tier_counts(cfg, n_tokens, n_forced, head_dim):
    """Number of candidate tokens (outside sinks/window) to store at 16, 8 and 4 bits."""
    n_cand = n_tokens - n_forced
    budget = resolve_budget(cfg, n_tokens, n_forced, head_dim)
    remaining = max(budget * n_tokens - n_forced, 0.0)  # in fp16-token units
    if cfg.method == "select":
        return min(n_cand, int(remaining)), 0, 0
    counts, left = [], n_cand
    for share, bits in zip(cfg.tier_split, (16, 8, 4)):
        n = min(left, int(remaining * share / bytes_ratio(bits, head_dim)))
        counts.append(n)
        left -= n
    return tuple(counts)


def ilf_diversified_scores(attn, lam, keep_fraction, tau_quantile=None):
    """attn: a_{i,l} as [L, batch, tokens].
    d_i     = sum_l 1[a_{i,l} > tau_l]
    ILF_i   = log((L + 1) / (d_i + 1))
    S_{i,l} = a_{i,l} (1 + lam * ILF_i)
    tau_l defaults to layer l's (1 - keep_fraction) quantile, i.e. "token i would be kept
    by plain attention selection in layer l"."""
    n_layers = attn.shape[0]
    q = tau_quantile if tau_quantile is not None else 1.0 - keep_fraction
    q = min(max(q, 0.0), 1.0)
    tau = torch.quantile(attn, q, dim=-1, keepdim=True)
    d = (attn > tau).sum(0)
    ilf = torch.log((n_layers + 1) / (d.float() + 1))
    return attn * (1 + lam * ilf), d, ilf


def allocate_bits(scores, n16, n8, n4):
    """scores [batch, tokens] -> bits [batch, tokens]: best n16 -> 16, next n8 -> 8, next n4 -> 4, rest 0."""
    b, m = scores.shape
    tiers = torch.zeros(m, dtype=torch.int16, device=scores.device)
    tiers[:n16] = 16
    tiers[n16:n16 + n8] = 8
    tiers[n16 + n8:n16 + n8 + n4] = 4
    order = scores.argsort(dim=-1, descending=True)
    return torch.zeros_like(order, dtype=torch.int16).scatter_(-1, order, tiers.expand(b, m).contiguous())


def make_config(name, method, budget=1.0, lam=0.0, **kw):
    return KVCompressionConfig(name, method, budget, lam, tier_split=TIER_SPLIT, sink_tokens=SINK_TOKENS,
                               window=OBS_WINDOW, pool_kernel=POOL_KERNEL, **kw)


def budget_tag(b):
    return b if isinstance(b, str) else f"{b:.0%}"


## 5. Compressed cache with packed mixed-precision storage
`CompressedLayer` plugs into the Transformers `DynamicCache`. It holds a 16-bit block plus packed
8-bit and 4-bit blocks (with metadata) for retained tokens; evicted tokens are simply absent.
`CompressedPrefix.make_cache()` builds a fresh cache that shares the stored tensors, so one compressed
prefix can serve many queries without copying.


In [ ]:
from transformers.cache_utils import DynamicCache, DynamicLayer


QUANT_GROUP = 32  # keys: per-channel over groups of 32 tokens (KIVI-style); values: per-token, 32 channels


@dataclass
class QTensor:
    """Packed asymmetric min-max quantization with fp16 scale/zero per group.
    axis="channel": each channel is quantized over a group of `group` tokens (used for keys,
    which have outlier channels); axis="token": each token over `group` channels (values).
    4-bit codes are packed two per uint8 byte, so storage really shrinks."""

    data: torch.Tensor   # uint8 [b, h, n_padded, d] (or d/2 when 4-bit)
    scale: torch.Tensor
    zero: torch.Tensor
    bits: int
    axis: str
    group: int
    num_tokens: int

    @property
    def nbytes(self):
        return sum(t.numel() * t.element_size() for t in (self.data, self.scale, self.zero))

    def dequantize(self, dtype):
        q = self.data
        if self.bits == 4:
            q = torch.stack((q & 0xF, q >> 4), dim=-1).flatten(-2)
        b, h, n, d = q.shape
        shape = (b, h, n // self.group, self.group, d) if self.axis == "channel" else (b, h, n, d // self.group, self.group)
        x = q.to(dtype).reshape(shape) * self.scale.to(dtype) + self.zero.to(dtype)
        return x.reshape(b, h, n, d)[:, :, :self.num_tokens]


def quantize(x, bits, axis, group=QUANT_GROUP):
    assert bits in (4, 8)
    b, h, n, d = x.shape
    xf = x.float()
    if axis == "channel":
        pad = (-n) % group  # pad by repeating the last token so it does not widen the range
        xf = torch.cat([xf, xf[:, :, -1:].expand(b, h, pad, d)], dim=2)
        xg, red = xf.reshape(b, h, (n + pad) // group, group, d), -2
    else:
        group = min(group, d)
        xg, red = xf.reshape(b, h, n, d // group, group), -1
    lo, hi = xg.amin(red, keepdim=True), xg.amax(red, keepdim=True)
    qmax = 2 ** bits - 1
    scale = ((hi - lo) / qmax).clamp_min(1e-5)
    q = torch.round((xg - lo) / scale).clamp_(0, qmax).to(torch.uint8).reshape(b, h, -1, d)
    if bits == 4:
        q = q[..., 0::2] | (q[..., 1::2] << 4)
    return QTensor(q.contiguous(), scale.half(), lo.half(), bits, axis, group, n)


class CompressedLayer(DynamicLayer):
    """One layer's retained KV entries: a 16-bit part plus packed 8/4-bit parts.
    Newly generated tokens are appended to the 16-bit part. Keys are stored post-RoPE,
    so evicted/reordered entries keep their original positions."""

    def __init__(self, keys16, values16, quantized=()):
        super().__init__()
        self.dtype, self.device = keys16.dtype, keys16.device
        self.keys, self.values = keys16, values16
        self.quantized = list(quantized)  # [(QTensor keys, QTensor values), ...]
        self.is_initialized = True

    def update(self, key_states, value_states, cache_kwargs=None):
        self.keys = torch.cat([self.keys, key_states], dim=-2)
        self.values = torch.cat([self.values, value_states], dim=-2)
        if not self.quantized:
            return self.keys, self.values
        keys = [qk.dequantize(self.dtype) for qk, _ in self.quantized] + [self.keys]
        values = [qv.dequantize(self.dtype) for _, qv in self.quantized] + [self.values]
        return torch.cat(keys, dim=-2), torch.cat(values, dim=-2)

    def get_seq_length(self):
        return self.keys.shape[-2] + sum(qk.num_tokens for qk, _ in self.quantized)


def storage_bytes(layers):
    """KV bytes with 16-bit entries counted at 2 bytes (independent of compute dtype)."""
    total = 0
    for k, v, quantized in layers:
        total += 2 * (k.numel() + v.numel())
        total += sum(qk.nbytes + qv.nbytes for qk, qv in quantized)
    return total


@dataclass
class CompressedPrefix:
    """A compressed, reusable prefix cache. `make_cache()` returns a fresh cache for each
    query, so one compressed prefix can serve many queries (prefix caching)."""

    layers: list   # per layer: (keys16, values16, [(QTensor, QTensor), ...])
    length: int    # number of original prefix tokens = next position id
    stats: dict

    def make_cache(self):
        cache = DynamicCache()
        cache.layers = [CompressedLayer(k, v, q) for k, v, q in self.layers]
        return cache


def _gather_tokens(x, idx):
    return x.gather(2, idx[:, None, :, None].expand(-1, x.shape[1], -1, x.shape[3]))


def _positions_with(bits_l, value):
    """Sorted positions holding `value` bits, [batch, n]; every batch row has the same count."""
    mask = bits_l == value
    n = int(mask[0].sum())
    return mask.nonzero()[:, 1].view(bits_l.shape[0], n) if n else None


def build_layer(keys, values, bits_l):
    idx16 = _positions_with(bits_l, 16)
    k16, v16 = _gather_tokens(keys, idx16), _gather_tokens(values, idx16)
    quantized = []
    for b in (8, 4):
        idx = _positions_with(bits_l, b)
        if idx is not None:
            quantized.append((quantize(_gather_tokens(keys, idx), b, "channel"),
                              quantize(_gather_tokens(values, idx), b, "token")))
    return k16, v16, quantized


## 6. Prefill → compress → continue
`prefill_and_compress` returns the compressed prefix and its statistics: bytes kept, the share of tokens at
each bit width, unique positions kept across layers, and the mean pairwise Jaccard overlap of
the layers' kept sets. `extend` and `greedy_generate` continue from it at the original positions.


In [ ]:
def model_dims(model):
    c = model.config
    head_dim = getattr(c, "head_dim", None) or c.hidden_size // c.num_attention_heads
    return c.num_hidden_layers, c.num_key_value_heads, head_dim


def selection_stats(bits, sink, window):
    """Diversity diagnostics from the bit plan bits [L, batch, N]."""
    n_layers, _, n_tokens = bits.shape
    cand = bits[..., sink:n_tokens - window] > 0  # candidate positions only (forced ones excluded)
    stats = {f"frac_tokens_{b}bit": (bits == b).float().mean().item() for b in (16, 8, 4, 0)}
    stats["tokens_retained_frac"] = (bits > 0).float().mean().item()
    if cand.shape[-1] == 0:
        return stats
    kept_any = cand.any(0).float()  # [batch, M]
    stats["unique_positions"] = kept_any.sum(-1).mean().item()
    stats["unique_positions_frac"] = kept_any.mean().item()
    stats["mean_kept_per_layer"] = cand.float().sum(-1).mean().item()
    r = cand.float()
    inter = torch.einsum("lbm,kbm->blk", r, r)
    sizes = r.sum(-1).T  # [batch, L]
    union = sizes[:, :, None] + sizes[:, None, :] - inter
    jac = inter / union.clamp_min(1)
    off_diag = ~torch.eye(n_layers, dtype=torch.bool, device=bits.device)
    stats["mean_layer_jaccard"] = jac[:, off_diag].mean().item()
    return stats


@torch.inference_mode()
def prefill_and_compress(model, input_ids, cfg):
    """Prefill `input_ids` [batch, N] (unpadded) and compress the resulting KV cache."""
    batch, n_tokens = input_ids.shape
    n_layers, n_kv_heads, head_dim = model_dims(model)
    need_scores = cfg.method in ("select", "mixed")
    t0 = time.perf_counter()
    cache = DynamicCache()
    if need_scores:
        RECORDER.start(cfg.window)
    try:
        out = model(input_ids=input_ids, past_key_values=cache, use_cache=True, logits_to_keep=1)
    finally:
        scores = RECORDER.stop()
    last_logits = out.logits[:, -1].float()
    full = [(layer.keys, layer.values) for layer in cache.layers]
    del out, cache

    sink = min(cfg.sink_tokens, n_tokens)
    window = min(cfg.window, n_tokens - sink)
    n_forced, n_cand = sink + window, n_tokens - sink - window
    bits = torch.full((n_layers, batch, n_tokens), 16, dtype=torch.int16, device=input_ids.device)
    if cfg.method != "full" and n_cand > 0:
        if need_scores:
            attn = torch.stack([scores[l] for l in range(n_layers)])  # a_{i,l}: [L, batch, N]
            if cfg.pool_kernel > 1:
                k = cfg.pool_kernel
                attn = F.avg_pool1d(attn.reshape(-1, 1, n_tokens), k, stride=1, padding=k // 2,
                                    count_include_pad=False).reshape(n_layers, batch, n_tokens)
            n16, n8, n4 = tier_counts(cfg, n_tokens, n_forced, head_dim)
            keep_fraction = (n16 + n8 + n4) / n_cand
            s, _, _ = ilf_diversified_scores(attn[..., sink:n_tokens - window], cfg.lam, keep_fraction,
                                             cfg.tau_quantile)
            bits[..., sink:n_tokens - window] = torch.stack([allocate_bits(s[l], n16, n8, n4)
                                                             for l in range(n_layers)])
        elif cfg.method == "quant":
            bits[..., sink:n_tokens - window] = cfg.quant_bits
        else:
            raise ValueError(f"unknown method {cfg.method!r}")
        layers = [build_layer(k, v, bits[l]) for l, (k, v) in enumerate(full)]
    else:
        layers = [(k, v, []) for k, v in full]
    del full
    if input_ids.is_cuda:
        torch.cuda.synchronize()

    full_bytes = n_layers * 2 * batch * n_kv_heads * n_tokens * head_dim * 2
    kv_bytes = storage_bytes(layers)
    stats = {
        "prefix_tokens": n_tokens,
        "kv_bytes": kv_bytes,
        "full_kv_bytes": full_bytes,
        "kv_bytes_pct": 100 * kv_bytes / full_bytes,
        "prefill_compress_s": time.perf_counter() - t0,
        **selection_stats(bits, sink, window),
    }
    return CompressedPrefix(layers, n_tokens, stats), last_logits


@torch.inference_mode()
def extend(model, cache, token_ids, start, all_logits=False):
    """Run new tokens [batch, n] on top of `cache`, at original positions start..start+n-1."""
    n = token_ids.shape[1]
    pos = torch.arange(start, start + n, device=token_ids.device)
    out = model(input_ids=token_ids, past_key_values=cache, use_cache=True,
                position_ids=pos[None].expand(token_ids.shape[0], -1), cache_position=pos,
                logits_to_keep=0 if all_logits else 1)
    return out.logits.float()


def eos_ids(model, tokenizer):
    ids = model.generation_config.eos_token_id
    ids = set(ids if isinstance(ids, (list, tuple)) else [ids])
    ids.add(tokenizer.eos_token_id)
    return {i for i in ids if i is not None}


@torch.inference_mode()
def greedy_generate(model, tokenizer, prefix, suffix_ids, max_new_tokens, stop=()):
    """Greedy decoding (batch 1) from a compressed prefix followed by `suffix_ids` [1, n]."""
    cache = prefix.make_cache()
    logits = extend(model, cache, suffix_ids, prefix.length)[:, -1]
    pos, out, stop_ids = prefix.length + suffix_ids.shape[1], [], eos_ids(model, tokenizer)
    for _ in range(max_new_tokens):
        nxt = int(logits.argmax(-1))
        if nxt in stop_ids:
            break
        out.append(nxt)
        if stop and any(s in tokenizer.decode(out, skip_special_tokens=True) for s in stop):
            break
        logits = extend(model, cache, torch.tensor([[nxt]], device=suffix_ids.device), pos)[:, -1]
        pos += 1
    text = tokenizer.decode(out, skip_special_tokens=True)
    for s in stop:
        text = text.split(s)[0]
    return text


## 7. Load the model
The model uses the `kv_compress` attention backend for every method, including the uncompressed
baseline, so every comparison runs on the same kernels.


In [ ]:
def load_model(model_id):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = (torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported()
             else torch.float16 if device == "cuda" else torch.float32)
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, attn_implementation="kv_compress")
    mdl = mdl.to(device).eval()
    max_len = min(MAX_INPUT_TOKENS, getattr(mdl.config, "max_position_embeddings", MAX_INPUT_TOKENS))
    print(f"{model_id} | {mdl.device} | {mdl.dtype} | layers/kv-heads/head-dim = {model_dims(mdl)} | max input {max_len}")
    return mdl, tok, max_len


model, tokenizer, MAX_LEN = load_model(MODEL_ID)


### Sanity checks and a look at ILF
1. With no compression, prefill + `extend` must reproduce the logits of one ordinary forward pass.
2. On a short context, compare the tokens kept by plain attention selection and by ILF selection in
   one layer, and check how much ILF raises the number of unique positions kept across layers.


In [ ]:
CONTEXT = (
    "The archive has three rooms. The red room holds maps. The blue room holds a brass key. "
    "The green room holds letters. The archivist, Ms. Okafor, locks the blue room every night at nine. "
    "Visitors must sign the ledger by the door and may not bring food inside. "
) * 3 + "Question: Which room holds the brass key? Answer: The"
ids = tokenizer(CONTEXT, return_tensors="pt").input_ids.to(model.device)
with torch.inference_mode():
    reference = model(ids).logits.float()
cut = ids.shape[1] - 4
prefix, _ = prefill_and_compress(model, ids[:, :cut], make_config("full", "full"))
logits = extend(model, prefix.make_cache(), ids[:, cut:], cut, all_logits=True)
print(f"max |logit difference| full-cache path vs plain forward: {(logits - reference[:, cut:]).abs().max():.2e}")

# ILF on this context, layer by layer
with torch.inference_mode():
    RECORDER.start(OBS_WINDOW)
    model(ids)
    scores = RECORDER.stop()
attn = torch.stack([scores[l] for l in range(len(scores))])[:, 0]  # [L, N]
cand = attn[:, SINK_TOKENS:-OBS_WINDOW]
_, d, ilf = ilf_diversified_scores(cand[:, None], LAMBDA, keep_fraction=0.25)
tokens = tokenizer.convert_ids_to_tokens(ids[0, SINK_TOKENS:-OBS_WINDOW])
print(f"\n{cand.shape[1]} candidate tokens, {cand.shape[0]} layers. Most and least 'common' tokens (d_i):")
order = d[0].argsort(descending=True)
print("  high d_i (low ILF):", [(tokens[i], int(d[0, i])) for i in order[:8]])
print("  low d_i (high ILF):", [(tokens[i], int(d[0, i])) for i in order[-8:]])

rows = []
for name, lam in [("attention (λ=0)", 0.0), (f"ILF (λ={LAMBDA})", LAMBDA)]:
    p, _ = prefill_and_compress(model, ids[:, :cut], make_config(name, "select", 0.35, lam))
    rows.append({"selection": name, **{k: round(p.stats[k], 3) for k in
                 ["kv_bytes_pct", "unique_positions", "unique_positions_frac", "mean_layer_jaccard"]},
                 "answer": greedy_generate(model, tokenizer, p, ids[:, cut:], 6, ["\n"])})
pd.DataFrame(rows)


## 8. Methods and matched budgets
Every row in the grid is a `KVCompressionConfig`. At each budget there are four ablation cells:
{attention, ILF} × {selection only, mixed precision}. They are compared against the full cache and
uniform int8/int4 quantization. The `@int4`/`@int8` budgets are byte-matched to the uniform-quantization rows.


In [ ]:
def method_grid(budgets=BUDGETS, lam=LAMBDA):
    """Ablation grid at matched memory budgets:
    attn-select vs ilf-select      -> effect of ILF diversification on selection
    *-select vs *-mixed            -> effect of mixed precision on top of selection
    quant-int4/8 vs *-mixed@int4/8 -> selection + mixed precision vs uniform quantization"""
    methods = [make_config("full", "full"),
               make_config("quant-int8", "quant", quant_bits=8),
               make_config("quant-int4", "quant", quant_bits=4)]
    for b in budgets:
        t = budget_tag(b)
        methods += [make_config(f"attn-select@{t}", "select", b, 0.0),
                    make_config(f"ilf-select@{t}", "select", b, lam),
                    make_config(f"attn-mixed@{t}", "mixed", b, 0.0),
                    make_config(f"ilf-mixed@{t}", "mixed", b, lam)]
    return methods


def describe(cfg):
    return {"method": cfg.name, "family": cfg.name.split("@")[0], "kind": cfg.method,
            "budget": str(cfg.budget), "lambda": cfg.lam}


def save_table(df, model_id, name):
    path = RESULTS_DIR / model_id.replace("/", "__") / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved {path}")


METHODS = method_grid()
print(f"{len(METHODS)} methods:", [m.name for m in METHODS])


## 9. Evaluation helpers
`CompressedKVLM` is an lm-evaluation-harness model that routes every request through the compressed
cache (details in its docstring). The other helpers run the retrieval and memory/latency experiments.


In [ ]:
from lm_eval import evaluator
from lm_eval.api.model import LM
from lm_eval.utils import make_table
from tqdm.auto import tqdm

# Where the question starts in each LongBench prompt. With QUERY_AWARE = False, everything
# before the marker is the reusable prefix: it is compressed *without seeing the question*,
# then the question is run on top of the compressed cache (prefix-caching setup).
LONGBENCH_QUESTION_MARKERS = [
    "\n\nAnswer the question based on the given passages.",          # hotpotqa, 2wikimqa, musique
    "\n\nNow, answer the following question based on the above text",  # multifieldqa_en
    "\n\n Answer the question based on the above article",            # qasper
    "\n\nNow, answer the question based on the story",                 # narrativeqa
    "\n\nThe following is an abstract.",                               # passage_retrieval_en
    "\n\nNow, write a one-page summary of the report.",                # gov_report
]


def middle_truncate(ids, max_len):
    """LongBench convention: keep the first and last halves of an over-long prompt."""
    if len(ids) <= max_len:
        return ids
    half = max_len // 2
    return ids[:half] + ids[len(ids) - (max_len - half):]


class CompressedKVLM(LM):
    """lm-evaluation-harness model that runs every request through a compressed KV cache.

    loglikelihood (MMLU): the context minus its last token is the prefix; it is compressed,
    then [last context token + continuation] is scored on top of the compressed cache, so
    every scored token depends on the compressed cache. Requests sharing a context reuse
    one compressed prefix.
    generate_until (LongBench): prefix = prompt up to the question (QUERY_AWARE=False) or
    the whole prompt minus its last token (QUERY_AWARE=True); greedy decoding afterwards."""

    def __init__(self, model, tokenizer, cfg, max_input_tokens=4096, query_aware=False,
                 markers=LONGBENCH_QUESTION_MARKERS):
        super().__init__()
        self.model, self.tokenizer, self.cfg = model, tokenizer, cfg
        self.max_input_tokens, self.query_aware, self.markers = max_input_tokens, query_aware, markers
        self.device = model.device
        self.compression_stats = []

    def _ids(self, text, special=True):
        return self.tokenizer(text, add_special_tokens=special).input_ids

    def _tensor(self, ids):
        return torch.tensor([ids], device=self.device)

    def _compress(self, prefix_ids):
        prefix, _ = prefill_and_compress(self.model, self._tensor(prefix_ids), self.cfg)
        self.compression_stats.append(prefix.stats)
        return prefix

    def loglikelihood(self, requests, disable_tqdm=False):
        groups = defaultdict(list)
        for i, req in enumerate(requests):
            context, continuation = req.args
            groups[context].append((i, continuation))
        results = [None] * len(requests)
        for context, items in tqdm(groups.items(), desc=f"loglikelihood [{self.cfg.name}]", disable=disable_tqdm):
            n_space = len(context) - len(context.rstrip())  # move trailing spaces into the continuation
            ctx = context[:len(context) - n_space] if n_space else context
            ctx_ids = self._ids(ctx)
            conts = []
            for i, cont in items:
                cont = context[len(ctx):] + cont
                conts.append((i, self._ids(ctx + cont)[len(ctx_ids):]))
            longest = max(len(c) for _, c in conts)
            ctx_ids = ctx_ids[-(self.max_input_tokens - longest):]
            if len(ctx_ids) < 2:
                raise ValueError("context too short to split into prefix + last token")
            prefix = self._compress(ctx_ids[:-1])
            for i, cont_ids in conts:
                feed = self._tensor([ctx_ids[-1]] + cont_ids[:-1])
                logits = extend(self.model, prefix.make_cache(), feed, prefix.length, all_logits=True)[0]
                logprobs = logits.log_softmax(-1)
                target = torch.tensor(cont_ids, device=logits.device)
                ll = logprobs.gather(-1, target[:, None]).sum().item()
                results[i] = (ll, bool((logits.argmax(-1) == target).all()))
        return results

    def _split(self, context):
        if not self.query_aware:
            cut = max(context.rfind(m) for m in self.markers)
            if cut > 0:
                return context[:cut], context[cut:]
        return None, None

    def generate_until(self, requests, disable_tqdm=False):
        outputs = []
        for req in tqdm(requests, desc=f"generate [{self.cfg.name}]", disable=disable_tqdm):
            context, gen_kwargs = req.args
            until = gen_kwargs.get("until") or []
            until = [until] if isinstance(until, str) else list(until)
            max_new = int(gen_kwargs.get("max_gen_toks", 128))
            budget = self.max_input_tokens - max_new
            prefix_str, suffix_str = self._split(context)
            if prefix_str is None:
                ids = middle_truncate(self._ids(context), budget)
                prefix_ids, suffix_ids = ids[:-1], ids[-1:]
            else:
                suffix_ids = self._ids(suffix_str, special=False)
                prefix_ids = middle_truncate(self._ids(prefix_str), budget - len(suffix_ids))
            prefix = self._compress(prefix_ids)
            outputs.append(greedy_generate(self.model, self.tokenizer, prefix, self._tensor(suffix_ids),
                                           max_new, until))
        return outputs

    def loglikelihood_rolling(self, requests, disable_tqdm=False):
        raise NotImplementedError("perplexity tasks are not used in this project")


def mean_compression_stats(stats_list):
    if not stats_list:
        return {}
    keys = [k for k in stats_list[0] if isinstance(stats_list[0][k], (int, float))]
    return {k: sum(s.get(k, 0) for s in stats_list) / len(stats_list) for k in keys}


def run_lm_eval(model, tokenizer, cfg, tasks, limit, num_fewshot=None, max_input_tokens=4096,
                query_aware=False):
    lm = CompressedKVLM(model, tokenizer, cfg, max_input_tokens, query_aware)
    results = evaluator.simple_evaluate(
        model=lm, tasks=tasks, num_fewshot=num_fewshot, limit=limit, log_samples=False,
        random_seed=42, numpy_random_seed=42, torch_random_seed=42, fewshot_random_seed=42,
    )
    return results, mean_compression_stats(lm.compression_stats)


In [ ]:
STAT_COLUMNS = ["kv_bytes_pct", "tokens_retained_frac", "unique_positions", "unique_positions_frac",
                "mean_layer_jaccard", "frac_tokens_16bit", "frac_tokens_8bit", "frac_tokens_4bit"]


def headline_metric(task_results):
    """Primary metric of one lm-eval task result dict (accuracy for MMLU, score for LongBench)."""
    for key in ("acc,none", "score,none", "exact_match,none"):
        if key in task_results:
            return key.split(",")[0], task_results[key]
    raise KeyError(f"no known metric in {list(task_results)}")


def run_benchmark_suite(model, tokenizer, methods, tasks, limit, num_fewshot=None, max_input_tokens=4096):
    rows = []
    for cfg in methods:
        results, stats = run_lm_eval(model, tokenizer, cfg, tasks, limit, num_fewshot,
                                     max_input_tokens, QUERY_AWARE)
        for task in tasks:
            metric, value = headline_metric(results["results"][task])
            rows.append({**describe(cfg), "task": task, "metric": metric, "value": value,
                         **{k: stats.get(k) for k in STAT_COLUMNS}})
        print(f"{cfg.name:>22}: " + ", ".join(f"{r['task']}={r['value']:.3f}" for r in rows[-len(tasks):]))
    return pd.DataFrame(rows)


# ---- Multi-query reuse retrieval: one compressed prefix, many questions about distant facts ----
NAMES = ["Alice", "Bruno", "Chiara", "Dmitri", "Esme", "Farid", "Greta", "Hiro", "Ines", "Jonas",
         "Kemal", "Lena", "Mateo", "Nadia", "Oskar", "Priya", "Quentin", "Rosa", "Sven", "Tariq",
         "Uma", "Viktor", "Wen", "Ximena", "Yusuf", "Zara", "Anouk", "Bodhi", "Celine", "Darius"]
FILLER = [
    "The morning shift checked the storage racks and found nothing unusual.",
    "A delivery truck arrived late because of heavy traffic near the bridge.",
    "Someone left a stack of old newspapers beside the photocopier again.",
    "The cafeteria served lentil soup, fresh bread and a small green salad.",
    "Maintenance replaced two flickering lights in the east corridor.",
    "The weekly meeting ran long while the team discussed the new schedule.",
    "Rain kept most visitors away from the courtyard during the afternoon.",
    "An inventory count showed the paper supply was slightly below target.",
    "The fire drill finished in under four minutes, a new building record.",
    "Several plants in the lobby were moved closer to the tall windows.",
    "The printer on the third floor jammed twice before lunch.",
    "A new coffee machine was installed and immediately drew a small crowd.",
]


def make_retrieval_doc(n_facts, n_sentences, seed):
    rng = random.Random(seed)
    names = rng.sample(NAMES, n_facts)
    codes = [rng.randint(10000, 99999) for _ in names]
    sentences = [rng.choice(FILLER) for _ in range(n_sentences)]
    slots = [int((j + 0.5) * n_sentences / n_facts) for j in range(n_facts)]
    for j in reversed(range(n_facts)):
        sentences.insert(slots[j], f"The secret code of {names[j]} is {codes[j]}.")
    doc = ("Below is a long building log. Secret codes for several people are hidden in it. "
           "Remember every one of them.\n\n" + " ".join(sentences) + "\n\n")
    facts = [{"name": n, "code": c, "depth": s / n_sentences} for n, c, s in zip(names, codes, slots)]
    return doc, facts


def retrieval_docs(tokenizer, n_docs, n_facts, n_tokens):
    per_sentence = len(tokenizer(" ".join(FILLER), add_special_tokens=False).input_ids) / len(FILLER)
    return [make_retrieval_doc(n_facts, int(n_tokens / per_sentence), seed) for seed in range(n_docs)]


def run_retrieval(model, tokenizer, methods, docs):
    """Compress each document ONCE, without any question in view, then ask about every fact."""
    rows = []
    for cfg in methods:
        for d, (doc, facts) in enumerate(docs):
            ids = torch.tensor([tokenizer(doc).input_ids], device=model.device)
            prefix, _ = prefill_and_compress(model, ids, cfg)
            for f in facts:
                q = f"Question: What is the secret code of {f['name']}?\nAnswer: The secret code of {f['name']} is"
                q_ids = torch.tensor([tokenizer(q, add_special_tokens=False).input_ids], device=model.device)
                answer = greedy_generate(model, tokenizer, prefix, q_ids, 8, ["\n"])
                digits = re.sub(r"\D", "", answer)[:5]
                rows.append({**describe(cfg), "doc": d, "depth": f["depth"],
                             "correct": digits == str(f["code"]),
                             # partial credit: fraction of the 5 digits right, in position
                             "digit_acc": sum(a == b for a, b in zip(digits, str(f["code"]))) / 5,
                             **{k: prefix.stats.get(k) for k in STAT_COLUMNS}})
        mine = [r for r in rows if r["method"] == cfg.name]
        print(f"{cfg.name:>22}: exact {sum(r['correct'] for r in mine) / len(mine):.3f}  "
              f"digit acc {sum(r['digit_acc'] for r in mine) / len(mine):.3f}  KV bytes {prefix.stats['kv_bytes_pct']:.1f}%")
    return pd.DataFrame(rows)


def summarize_retrieval(df):
    keys = ["method", "family", "kind", "budget", "lambda"]
    return (df.groupby(keys, sort=False)
              .agg(accuracy=("correct", "mean"), digit_acc=("digit_acc", "mean"), **{c: (c, "mean") for c in STAT_COLUMNS})
              .reset_index())


# ---- Memory and decode latency at a fixed workload (batch x sequence length) ----
def benchmark_memory_latency(model, tokenizer, cfg, batch, seq_len, decode_tokens, warmup=3):
    ids = []
    for s in range(batch):
        doc, _ = make_retrieval_doc(20, seq_len // 10, seed=1000 + s)
        t = tokenizer(doc).input_ids
        while len(t) < seq_len:
            t = t + t
        ids.append(t[:seq_len])
    input_ids = torch.tensor(ids, device=model.device)
    cuda = input_ids.is_cuda
    gc.collect()
    if cuda:
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        base = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()
    prefix, logits = prefill_and_compress(model, input_ids, cfg)
    nxt = logits.argmax(-1, keepdim=True)
    del logits
    row = {**describe(cfg), "batch": batch, "seq_len": seq_len, "kv_bytes_pct": prefix.stats["kv_bytes_pct"],
           "kv_MiB_logical": prefix.stats["kv_bytes"] / 2**20,
           "prefill_compress_ms": 1000 * prefix.stats["prefill_compress_s"]}
    if cuda:
        torch.cuda.synchronize()
        row["kv_MiB_measured"] = (torch.cuda.memory_allocated() - base) / 2**20
        row["prefill_peak_MiB"] = (torch.cuda.max_memory_allocated() - base) / 2**20
        torch.cuda.reset_peak_memory_stats()
    cache, pos, times = prefix.make_cache(), seq_len, []
    for step in range(decode_tokens + warmup):
        if cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        nxt = extend(model, cache, nxt, pos)[:, -1].argmax(-1, keepdim=True)
        if cuda:
            torch.cuda.synchronize()
        if step >= warmup:
            times.append(time.perf_counter() - t0)
        pos += 1
    row["decode_ms_per_token"] = 1000 * sum(times) / len(times)
    if cuda:
        row["decode_peak_MiB"] = (torch.cuda.max_memory_allocated() - base) / 2**20
    del cache, prefix
    return row


In [ ]:
import matplotlib.pyplot as plt

# Fixed colour + marker per method family (marker shape is a second, colour-independent cue).
FAMILY_STYLE = {
    "attn-select": ("#2a78d6", "o"),
    "ilf-select": ("#eb6834", "s"),
    "attn-mixed": ("#1baf7a", "^"),
    "ilf-mixed": ("#eda100", "D"),
    "uniform-quant": ("#e87ba4", "v"),
}


def plot_family(row):
    return "uniform-quant" if row["kind"] == "quant" else row["family"]


def tradeoff_plot(df, y, ylabel, title, path=None):
    """Quality vs KV bytes kept: one line per method family, full cache as a reference line."""
    df = df.assign(plot_family=df.apply(plot_family, axis=1))
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    full = df[df["kind"] == "full"]
    if len(full):
        ax.axhline(full[y].mean(), color="#8a8a85", lw=1.5, ls="--", label="full cache")
    for fam, (color, marker) in FAMILY_STYLE.items():
        part = df[df["plot_family"] == fam].sort_values("kv_bytes_pct")
        if len(part):
            ax.plot(part["kv_bytes_pct"], part[y], color=color, marker=marker, lw=2, ms=8, label=fam)
    ax.set_xlabel("KV-cache bytes kept (% of fp16)")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")
    ax.grid(alpha=0.25, lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()


def lambda_plot(df, y, ylabel, title, path=None):
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for fam, (color, marker) in FAMILY_STYLE.items():
        part = df[df["family"] == fam].sort_values("lambda")
        if len(part):
            ax.plot(part["lambda"], part[y], color=color, marker=marker, lw=2, ms=8, label=fam)
    ax.set_xlabel("λ (ILF weight)")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")
    ax.grid(alpha=0.25, lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()


## 10. MMLU: general reasoning
For loglikelihood tasks, the context minus its last token is compressed, and the last context token plus
the answer are scored on top of the compressed cache. Without this split, single-token answers would
be scored from uncompressed prefill logits, and compression would have no effect.
`MMLU_LIMIT` applies per subject.


In [ ]:
mmlu_df = run_benchmark_suite(model, tokenizer, METHODS, MMLU_TASKS, MMLU_LIMIT, MMLU_FEWSHOT, MAX_LEN)
save_table(mmlu_df, MODEL_ID, "mmlu")
mmlu_df[["method", "task", "value", "kv_bytes_pct", "unique_positions_frac"]]


## 11. LongBench: long-context tasks
Prompts longer than `MAX_INPUT_TOKENS` are middle-truncated (LongBench convention). Scores are the
official LongBench metrics (QA-F1 or retrieval accuracy).


In [ ]:
longbench_df = run_benchmark_suite(model, tokenizer, METHODS, LONGBENCH_TASKS, LONGBENCH_LIMIT, None, MAX_LEN)
save_table(longbench_df, MODEL_ID, "longbench")
longbench_avg = (longbench_df.groupby(["method", "family", "kind", "budget", "lambda"], sort=False)
                 .agg(score=("value", "mean"), **{c: (c, "mean") for c in STAT_COLUMNS}).reset_index())
display(longbench_df.pivot(index="method", columns="task", values="value").loc[[m.name for m in METHODS]])
tradeoff_plot(longbench_avg, "score", "LongBench score (mean over tasks)", f"LongBench · {MODEL_ID}",
              RESULTS_DIR / MODEL_ID.replace("/", "__") / "longbench_tradeoff.png")


## 12. Long-range retrieval with a reused prefix
This is the experiment closest to the proposal's motivation. A long log hides `RETRIEVAL_FACTS` secret
codes at evenly spaced depths. Each log is **compressed once, before any question is seen**, and then
the same compressed prefix answers a question about every fact. A method that keeps only the tokens one
query cares about will fail on the others. We report exact match and digit accuracy (the fraction of the 5 digits right, in position). Digit
accuracy gives partial credit, because Qwen tokenizes each digit separately and exact match drops to 0
quickly under aggressive, query-agnostic eviction. Accuracy is also broken down by the fact's depth in the document.


In [ ]:
docs = retrieval_docs(tokenizer, RETRIEVAL_DOCS, RETRIEVAL_FACTS, min(RETRIEVAL_TOKENS, MAX_LEN - 64))
retrieval_df = run_retrieval(model, tokenizer, METHODS, docs)
save_table(retrieval_df, MODEL_ID, "retrieval_raw")
retrieval_summary = summarize_retrieval(retrieval_df)
save_table(retrieval_summary, MODEL_ID, "retrieval")
display(retrieval_summary[["method", "accuracy", "digit_acc", "kv_bytes_pct", "unique_positions_frac", "mean_layer_jaccard"]])
display(retrieval_df.assign(depth_bin=pd.cut(retrieval_df["depth"], [0, .25, .5, .75, 1]))
        .pivot_table(index="method", columns="depth_bin", values="digit_acc", aggfunc="mean", observed=False)
        .loc[[m.name for m in METHODS]])
tradeoff_plot(retrieval_summary, "digit_acc", "retrieval digit accuracy", f"Reused-prefix retrieval · {MODEL_ID}",
              RESULTS_DIR / MODEL_ID.replace("/", "__") / "retrieval_tradeoff.png")


## 13. Memory and decode latency
The proposal's workload is batch 4 × 4096 tokens. For each method:
* **kv_bytes_pct / kv_MiB_logical**: stored KV bytes, including quantization metadata.
* **kv_MiB_measured**: GPU memory still allocated after prefill + compression, i.e. the real cache
  footprint (model weights excluded).
* **prefill_peak_MiB / decode_peak_MiB**: peak extra memory. Prefill builds the full cache before
  compressing, so its peak is about the same for every method. Decode peak includes the transient
  dequantized copy of one layer.
* **decode_ms_per_token**: mean wall time per decoding step (whole batch), CUDA-synchronized, after warm-up.
* **prefill_compress_ms**: prefill plus scoring plus compression time.


In [ ]:
bench_df = pd.DataFrame([benchmark_memory_latency(model, tokenizer, cfg, BENCH_BATCH, BENCH_SEQ_LEN, BENCH_DECODE_TOKENS)
                         for cfg in METHODS])
save_table(bench_df, MODEL_ID, "memory_latency")
bench_df.drop(columns=["family", "kind"]).round(2)


## 14. Diversity and the λ sweep
**Diversity at matched budgets.** For selection methods, how many distinct token positions survive in
*at least one* layer (`unique_positions_frac`), and how similar the layers' kept sets are
(`mean_layer_jaccard`; lower means more diverse). This directly tests whether ILF reduces token
over-concentration.

**λ sweep.** Retrieval accuracy and diversity as the ILF weight grows, at the first numeric budget.


In [ ]:
diversity = retrieval_summary[retrieval_summary["kind"].isin(["select", "mixed"])]
display(diversity[["method", "budget", "lambda", "unique_positions", "unique_positions_frac",
                   "mean_layer_jaccard", "tokens_retained_frac", "accuracy", "digit_acc"]])

sweep_budget = next(b for b in BUDGETS if not isinstance(b, str))
sweep_methods = [make_config(f"ilf-{kind}@{budget_tag(sweep_budget)} λ={lam}", kind, sweep_budget, lam)
                 for kind in ("select", "mixed") for lam in LAMBDA_SWEEP]
sweep_df = summarize_retrieval(run_retrieval(model, tokenizer, sweep_methods, docs))
save_table(sweep_df, MODEL_ID, "lambda_sweep")
display(sweep_df[["method", "lambda", "accuracy", "digit_acc", "unique_positions_frac", "mean_layer_jaccard", "kv_bytes_pct"]])
out = RESULTS_DIR / MODEL_ID.replace("/", "__")
lambda_plot(sweep_df, "digit_acc", "retrieval digit accuracy", f"λ sweep at {budget_tag(sweep_budget)} budget", out / "lambda_accuracy.png")
lambda_plot(sweep_df, "unique_positions_frac", "unique positions kept (fraction)",
            f"Diversity vs λ at {budget_tag(sweep_budget)} budget", out / "lambda_diversity.png")


## 15. Summary table
One row per method, joining every experiment run above for the current model.


In [ ]:
def summary_table(mmlu_df, longbench_df, retrieval_summary, bench_df):
    s = retrieval_summary[["method", "kv_bytes_pct", "unique_positions_frac", "mean_layer_jaccard", "accuracy", "digit_acc"]]
    s = s.rename(columns={"accuracy": "retrieval_exact", "digit_acc": "retrieval_digit_acc"})
    s = s.merge(mmlu_df.groupby("method", sort=False)["value"].mean().rename("mmlu_acc").reset_index(), on="method")
    s = s.merge(longbench_df.groupby("method", sort=False)["value"].mean().rename("longbench").reset_index(), on="method")
    cols = ["method"] + [c for c in ["kv_MiB_measured", "decode_ms_per_token"] if c in bench_df]
    return s.merge(bench_df[cols], on="method")


summary = summary_table(mmlu_df, longbench_df, retrieval_summary, bench_df)
save_table(summary, MODEL_ID, "summary")
summary.round(3)


## 16. Other models (Qwen 2.5 sizes, Llama 2, Llama 3)
Runs the whole suite for each model in `EXTRA_MODEL_IDS`, freeing GPU memory in between. Large models
need a bigger GPU (an A100 for 7–8B with 4k contexts) and a lot more time, so you may want to shrink
`METHODS` or the limits first. Llama-2's 4096-token context limits `MAX_LEN` automatically.


In [ ]:
RUN_EXTRA_MODELS = False

def run_suite(model_id):
    mdl, tok, max_len = load_model(model_id)
    try:
        mm = run_benchmark_suite(mdl, tok, METHODS, MMLU_TASKS, MMLU_LIMIT, MMLU_FEWSHOT, max_len)
        lb = run_benchmark_suite(mdl, tok, METHODS, LONGBENCH_TASKS, LONGBENCH_LIMIT, None, max_len)
        dc = retrieval_docs(tok, RETRIEVAL_DOCS, RETRIEVAL_FACTS, min(RETRIEVAL_TOKENS, max_len - 64))
        rs = summarize_retrieval(run_retrieval(mdl, tok, METHODS, dc))
        bn = pd.DataFrame([benchmark_memory_latency(mdl, tok, c, BENCH_BATCH, min(BENCH_SEQ_LEN, max_len - BENCH_DECODE_TOKENS - 8),
                                                    BENCH_DECODE_TOKENS) for c in METHODS])
        for df, name in [(mm, "mmlu"), (lb, "longbench"), (rs, "retrieval"), (bn, "memory_latency")]:
            save_table(df, model_id, name)
        s = summary_table(mm, lb, rs, bn).assign(model=model_id)
        save_table(s, model_id, "summary")
        return s
    finally:
        del mdl
        gc.collect()
        torch.cuda.empty_cache()


if RUN_EXTRA_MODELS:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    all_summaries = pd.concat([summary.assign(model=MODEL_ID)] + [run_suite(m) for m in EXTRA_MODEL_IDS])
    all_summaries.to_csv(RESULTS_DIR / "summary_all_models.csv", index=False)
    display(all_summaries.round(3))


**Limitations to state in the write-up.** Batches are unpadded (equal-length inputs). Quantized tokens
are dequantized every decode step in PyTorch, not with a fused kernel, so the latency numbers include
that overhead. The prefill peak is not reduced, because compression happens after prefill. MMLU
contexts are short, so most of the prompt sits inside the protected window.

References: [Transformers cache guide](https://huggingface.co/docs/transformers/v4.57.1/en/cache_explanation),
[Attention interface](https://huggingface.co/docs/transformers/v4.57.1/en/attention_interface),
[LM Evaluation Harness API](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/docs/python-api.md),
[LongBench tasks](https://github.com/EleutherAI/lm-evaluation-harness/blob/main/lm_eval/tasks/longbench/README.md),
SnapKV (Li et al., 2024), KIVI (Liu et al., 2024).
